## Identifier Frequency Analysis

Compare identifier distribution patterns between AI-generated and human-authored code.

### Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import ast
import os
import builtins
from collections import Counter
from pathlib import Path

### Configure Paths and Parameters

In [ ]:
BASE_DIR = "/home/fahad/Documents/MASTERS/CODEBLOCKS/Masters-Thesis-Code/promptMark"
BASE_MODEL = 'gemini'

AI_OUTPUT_DIR = f"{BASE_DIR}/output/EXP_2_V1_V2/gemini_exp2_during_gen_v2_100_mbpp"
HUMAN_CODE_DIR = f"{BASE_DIR}/datasets/core/sanitized-mbpp-sample-100-codes"

AI_CSV_OUTPUT = f"{BASE_DIR}/results/indentifier-analysis/{BASE_MODEL}_generated_identifier_frequency.csv"
HUMAN_CSV_OUTPUT = f"{BASE_DIR}/results/indentifier-analysis/human_authored_identifier_frequency.csv"

similarity_output = f"{BASE_DIR}/results/indentifier-analysis/{BASE_MODEL}_identifier_similarity_analysis.csv"


print(f"AI Output Directory: {AI_OUTPUT_DIR}")
print(f"Human Code Directory: {HUMAN_CODE_DIR}")
print(f"AI Results CSV: {AI_CSV_OUTPUT}")
print(f"Human Results CSV: {HUMAN_CSV_OUTPUT}")

### Extract Tokens from Code

In [ ]:
COMMON_STD_METHODS = {
    "self", "re", "cls", "append", "join", "dummy_function", "find", "kwargs", "args",
    "range", "print", "len", "dict", "list", "str", "int", "float", "set", "tuple",
    "os", "np", "subprocess", "now", "today", "timedelta", "strptime", "date", "time",
    "datetime", "logging", "log", "info", "debug", "error", "warning", "exception",
    "lower", "upper", "strip", "split", "replace", "endswith", "startswith", "extend",
    "insert", "remove", "pop", "sort", "clear", "keys", "values", "items", "get",
    "update", "copy", "format", "count", "index",
}

BUILTIN_NAMES = set(dir(builtins)).union(COMMON_STD_METHODS)

class CodeNavigator(ast.NodeVisitor):
    def __init__(self):
        self.public_classes = set()
        self.non_public_classes = set()
        self.public_funcs = set()
        self.non_public_funcs = set()
        self.public_vars = set()
        self.non_public_vars = set()

    def visit_FunctionDef(self, node):
        name = node.name
        if name.startswith("__") and name.endswith("__"):
            pass
        elif name.startswith("_"):
            self.non_public_funcs.add(name)
        else:
            self.public_funcs.add(name)

        for arg in node.args.args:
            if arg.arg not in BUILTIN_NAMES:
                self.non_public_vars.add(arg.arg)
        self.generic_visit(node)

    def visit_ClassDef(self, node):
        if node.name.startswith("_"):
            self.non_public_classes.add(node.name)
        else:
            self.public_classes.add(node.name)
        self.non_public_vars.add(node.name)
        self.generic_visit(node)

    def visit_Assign(self, node):
        for target in node.targets:
            if isinstance(target, ast.Name) and target.id not in BUILTIN_NAMES:
                if target.id.isupper():
                    self.public_vars.add(target.id)
                else:
                    self.non_public_vars.add(target.id)
        self.generic_visit(node)

    def visit_Attribute(self, node):
        if isinstance(node.value, ast.Name) and node.value.id == "self":
            if node.attr not in BUILTIN_NAMES and node.attr not in COMMON_STD_METHODS:
                self.public_funcs.add(node.attr)
        elif node.attr not in BUILTIN_NAMES and node.attr not in COMMON_STD_METHODS:
            self.non_public_vars.add(node.attr)
        self.generic_visit(node)

    def visit_Name(self, node):
        if node.id not in BUILTIN_NAMES:
            self.non_public_vars.add(node.id)
        self.generic_visit(node)

def extract_tokens(code):
    try:
        tree = ast.parse(code)
    except SyntaxError:
        return set()

    visitor = CodeNavigator()
    visitor.visit(tree)

    all_tokens = (
        visitor.public_classes
        | visitor.non_public_funcs
        | visitor.non_public_vars
        | visitor.non_public_classes
        | visitor.public_funcs
        | visitor.public_vars
    )

    cleaned_tokens = {
        t for t in all_tokens if t not in COMMON_STD_METHODS and t not in BUILTIN_NAMES
    }

    return cleaned_tokens

### Process AI-Generated Code

In [ ]:
ai_token_counter = Counter()
ai_output_path = Path(AI_OUTPUT_DIR)

if ai_output_path.exists():
    py_files = list(ai_output_path.glob("*.py"))
    print(f"Found {len(py_files)} AI-generated code files")
    
    for file_path in py_files:
        with open(file_path, 'r', encoding='utf-8') as f:
            code = f.read()
        tokens = extract_tokens(code)
        ai_token_counter.update(tokens)
    
    print(f"Extracted {len(ai_token_counter)} unique tokens from AI-generated code")
    print(f"Total token occurrences: {sum(ai_token_counter.values())}")
else:
    print(f"AI Output Directory not found: {AI_OUTPUT_DIR}")

ai_df = pd.DataFrame(
    ai_token_counter.most_common(),
    columns=['identifier', 'frequency']
)
print(f"\nAI-Generated Code - Top 10 identifiers:")
print(ai_df.head(10))

### Process Human-Authored Code

In [ ]:
human_token_counter = Counter()
human_code_path = Path(HUMAN_CODE_DIR)

if human_code_path.exists():
    py_files = list(human_code_path.glob("*.py"))
    print(f"Found {len(py_files)} human-authored code files")
    
    for file_path in py_files:
        with open(file_path, 'r', encoding='utf-8') as f:
            code = f.read()
        tokens = extract_tokens(code)
        human_token_counter.update(tokens)
    
    print(f"Extracted {len(human_token_counter)} unique tokens from human-authored code")
    print(f"Total token occurrences: {sum(human_token_counter.values())}")
else:
    print(f"Human Code Directory not found: {HUMAN_CODE_DIR}")

human_df = pd.DataFrame(
    human_token_counter.most_common(),
    columns=['identifier', 'frequency']
)
print(f"\nHuman-Authored Code - Top 10 identifiers:")
print(human_df.head(10))

### Save Results to CSV

In [ ]:
output_dir = Path(AI_CSV_OUTPUT).parent
output_dir.mkdir(parents=True, exist_ok=True)

ai_df_sorted = ai_df.sort_values('frequency', ascending=False).reset_index(drop=True)
human_df_sorted = human_df.sort_values('frequency', ascending=False).reset_index(drop=True)

ai_df_sorted.to_csv(AI_CSV_OUTPUT, index=False)
human_df_sorted.to_csv(HUMAN_CSV_OUTPUT, index=False)

print(f"✅ AI-Generated identifiers saved to: {AI_CSV_OUTPUT}")
print(f"   Total rows: {len(ai_df_sorted)}")
print(f"\n✅ Human-Authored identifiers saved to: {HUMAN_CSV_OUTPUT}")
print(f"   Total rows: {len(human_df_sorted)}")
print(f"\n📊 Summary:")
print(f"   AI-Generated: {len(ai_df_sorted)} unique identifiers, {ai_df_sorted['frequency'].sum()} total occurrences")
print(f"   Human-Authored: {len(human_df_sorted)} unique identifiers, {human_df_sorted['frequency'].sum()} total occurrences")

### Find Similar Identifiers Using Cosine Similarity

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import json

ai_identifiers = ai_df_sorted['identifier'].tolist()
human_identifiers = human_df_sorted['identifier'].tolist()

vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(2, 2))
all_identifiers = ai_identifiers + human_identifiers
tfidf_matrix = vectorizer.fit_transform(all_identifiers)

ai_tfidf = tfidf_matrix[:len(ai_identifiers)]
human_tfidf = tfidf_matrix[len(ai_identifiers):]

similarity_matrix = cosine_similarity(ai_tfidf, human_tfidf)

similar_pairs = []
for i, ai_id in enumerate(ai_identifiers):
    best_match_idx = similarity_matrix[i].argmax()
    similarity_score = float(similarity_matrix[i, best_match_idx])
    
    # if similarity_score > 0.3:
    human_id = human_identifiers[best_match_idx]
    similar_pairs.append({
        'ai_identifier': ai_id,
        'human_identifier': human_id,
        'similarity_score': round(similarity_score, 4),
        'ai_frequency': int(ai_df_sorted[ai_df_sorted['identifier'] == ai_id]['frequency'].values[0]),
        'human_frequency': int(human_df_sorted[human_df_sorted['identifier'] == human_id]['frequency'].values[0])
    })

similar_df = pd.DataFrame(similar_pairs).sort_values('similarity_score', ascending=False)

similar_df.to_csv(similarity_output, index=False)

print(f"✅ Similar identifiers analysis saved to: {similarity_output}")
print(f"   Found {len(similar_df)} identifier pairs with similarity > 0.3")
print(f"\nTop 10 most similar identifier pairs:")
print(similar_df.head(10))

### Visualize Similarity Trends

### Diagnostic: Understanding TPR/FPR Calculation Differences

In [ ]:
"""
ROOT CAUSE ANALYSIS: Why TPR differs between Method 1 and Method 2

Method 1 (calculate_auroc):
  - Uses continuous SCORES: -log10(p_unified)
  - Calculates ROC curve from scores (threshold-independent)
  - TPR shown is at a specific FPR threshold, NOT at p_threshold=0.05

Method 2 (notebook):
  - Uses binary PREDICTIONS: p_unified < 0.05
  - Applies fixed threshold to p_values FIRST
  - Calculates confusion matrix from binary decisions
  - Only shows metrics for that ONE threshold

THEY ARE NOT COMPARABLE without understanding the threshold!
"""

print("="*80)
print("ROOT CAUSE: Method 1 ≠ Method 2")
print("="*80)

print("\n1️⃣  METHOD 1 APPROACH (calculate_auroc):")
print("   Input: continuous scores (-log10(p_unified))")
print("   Process: ROC curve calculation")
print("   Output: TPR at FPR=0.0, 0.01, 0.05, 0.10 (from ROC)")
print("   NOTE: These thresholds on SCORES, not on p-values")

print("\n2️⃣  METHOD 2 APPROACH (notebook):")
print("   Input: binary predictions (p_unified < 0.05)")
print("   Process: confusion matrix at fixed threshold 0.05 on p-values")
print("   Output: TPR for ONLY this single operating point")
print("   NOTE: This is NOT from ROC curve!")

print("\n3️⃣  KEY INSIGHT:")
print("   ROC curve shows ALL possible (FPR, TPR) points")
print("   Method 2 shows only ONE point on that curve")
print("   Method 2's TPR=7 might be at FPR≠0.0")

print("\n4️⃣  FIX TO COMPARE THEM:")
print("   Use Method 1's ROC curve to find TPR at your desired FPR")
print("   OR use Method 2's threshold logic for ALL scores in Method 1")
print("   But DON'T mix continuous scores with binary p-value threshold!")

print("\n" + "="*80)
print("EXAMPLE: Why you get different values")
print("="*80)
print(f"""
Assume 100 samples total:
- 50 original (should have low scores)
- 50 generated (should have high scores)

With threshold=0.05 on p-values (Method 2):
  - Maybe only 7 out of 50 generated samples pass p < 0.05
  - TPR = 7/50 = 0.14 (14%)
  
But on ROC curve at FPR=0.0 (Method 1):
  - At FPR=0.0, ALL original samples must be correctly classified (0 false positives)
  - This requires a HIGHER threshold on scores
  - Maybe 22 out of 50 generated pass this higher threshold
  - TPR = 22/50 = 0.44 (44%)

Different thresholds → Different TPR values!
Both are correct, but for DIFFERENT operating points.
""")

In [ ]:
similar_df.head()

In [ ]:
# sort df by similarity_score in descending order
similar_df = similar_df.sort_values(by='similarity_score', ascending=True)
similar_df.head()

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

ax1 = axes[0, 0]
similarity_scores = similar_df['similarity_score'].values
ax1.hist(similarity_scores, bins=30, color='steelblue', edgecolor='black', alpha=0.7)
ax1.set_xlabel('Similarity Score')
ax1.set_ylabel('Frequency')
ax1.set_title('Distribution of Identifier Similarity Scores')
ax1.axvline(similarity_scores.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {similarity_scores.mean():.3f}')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2 = axes[0, 1]
top_20 = similar_df.head(20)
x_pos = range(len(top_20))
colors = plt.cm.RdYlGn(top_20['similarity_score'].values)
ax2.barh(x_pos, top_20['similarity_score'].values, color=colors)
ax2.set_yticks(x_pos)
ax2.set_yticklabels([f"{row['ai_identifier']} → {row['human_identifier']}" for _, row in top_20.iterrows()], fontsize=8)
ax2.set_xlabel('Similarity Score')
ax2.set_title('Top 20 Most Similar Identifier Pairs')
ax2.invert_yaxis()
ax2.grid(True, alpha=0.3, axis='x')

ax3 = axes[1, 0]
top_20_ai = similar_df.head(20)
ax3.scatter(top_20_ai['ai_frequency'], top_20_ai['similarity_score'], s=100, alpha=0.6, color='blue', label='AI Freq')
ax3_twin = ax3.twiny()
ax3_twin.scatter(top_20_ai['human_frequency'], top_20_ai['similarity_score'], s=100, alpha=0.6, color='orange', label='Human Freq', marker='^')
ax3.set_xlabel('AI Identifier Frequency', color='blue')
ax3_twin.set_xlabel('Human Identifier Frequency', color='orange')
ax3.set_ylabel('Similarity Score')
ax3.set_title('Similarity Score vs Token Frequency (Top 20 Pairs)')
ax3.grid(True, alpha=0.3)
ax3.legend(loc='upper left')
ax3_twin.legend(loc='upper right')

ax4 = axes[1, 1]
score_bins = [0, 0.2, 0.4, 0.6, 0.8, 1.0]
binned_data = pd.cut(similar_df['similarity_score'], bins=score_bins)
bin_counts = binned_data.value_counts().sort_index()
ax4.bar(range(len(bin_counts)), bin_counts.values, color='teal', alpha=0.7, edgecolor='black')
ax4.set_xticks(range(len(bin_counts)))
ax4.set_xticklabels([f'{score_bins[i]}-{score_bins[i+1]}' for i in range(len(score_bins)-1)], rotation=45)
ax4.set_xlabel('Similarity Score Range')
ax4.set_ylabel('Count')
ax4.set_title('Identifier Pairs by Similarity Range')
ax4.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(f"{BASE_DIR}/results/others/similarity_trends_visualization.png", dpi=300, bbox_inches='tight')
plt.show()

print(f"✅ Visualization saved to: {BASE_DIR}/results/others/similarity_trends_visualization.png")
print(f"\n📊 Similarity Score Statistics:")
print(f"   Mean: {similarity_scores.mean():.4f}")
print(f"   Median: {np.median(similarity_scores):.4f}")
print(f"   Std Dev: {similarity_scores.std():.4f}")
print(f"   Min: {similarity_scores.min():.4f}")
print(f"   Max: {similarity_scores.max():.4f}")